# Revisão do M5

Este notebook audita o M5 atual e define uma trajetória qualitativa de fricção de coordenação reproduzível para a RQ2.

**Conclusão principal:** três escores LLM agregados e constantes não formam uma trajetória analisável. A substituição proposta calcula densidade e composição de marcadores de fricção diretamente no corpus global de cada marco, com cobertura declarada e sem novas chamadas LLM. Registros e sessões de transcrição servem apenas para rastreabilidade, não como observações independentes.

## Estrutura recomendada

| Saída global por marco | Pergunta respondida | Relação com outras métricas |
|---|---|---|
| **M5a - Densidade de evidência** | Quantos marcadores de fricção ocorrem por mil tokens do corpus? | Qualitativa; não repete commits de M3/M4 |
| **M5b - Composição da fricção** | Qual proporção dos marcadores pertence a alinhamento, repasse, integração, bloqueio ou retrabalho? | Explica conteúdo, não volume de atividade |
| **M5c - Cobertura do corpus** | Quantos registros, sessões, tokens e marcadores fundamentam cada marco? | Audita comparabilidade, não mede desempenho |

M3 permanece responsável por concentração/autoria e M4 por magnitude de mudança limpa. M5 usa um corpus global por marco; não produz resultados por equipe, semestre ou sessão.

## 1. Vínculo com a RQ2

> **RQ2:** How does the temporal density of repository activity contrast with the qualitative typification of human coordination friction across the project lifecycle?

O notebook extrai a vinculação diretamente do texto atual do paper e falha se M5 não estiver dentro da subseção RQ2.

In [ ]:
from hashlib import sha256
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "paper_v8/latex_code/main.tex").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing paper_v8/latex_code/main.tex")


PROJECT_ROOT = find_project_root(Path.cwd())
PAPER_PATH = PROJECT_ROOT / "paper_v8/latex_code/main.tex"
LEGACY_M5_PATH = PROJECT_ROOT / "paper_v8/data/m5_coordination_friction_trajectory.csv"
LEGACY_M5_SOURCE_PATH = PROJECT_ROOT / "paper_v4/advanced_metrics/outputs/cohort_temporal_friction.csv"
TRANSCRIPTS_PATH = PROJECT_ROOT / "data/lake/transcript_sessions.parquet"

paper_text = PAPER_PATH.read_text(encoding="utf-8")
rq_matches = dict(re.findall(r"\\item \\textbf\{(RQ\d)[^}]*:\}\s*(.*?)\n", paper_text))
m5_position = paper_text.index(r"\textbf{M5 --")
rq2_position = paper_text.index(r"\subsubsection{RQ2:")
rq3_position = paper_text.index(r"\subsubsection{RQ3:")
detected_rq = "RQ2" if rq2_position < m5_position < rq3_position else "UNKNOWN"
assert detected_rq == "RQ2"
assert rq_matches.get(detected_rq)

pd.DataFrame([{
    "metric": "M5",
    "detected_rq": detected_rq,
    "rq_text": rq_matches[detected_rq],
    "paper_sha256": sha256(paper_text.encode("utf-8")).hexdigest(),
}])

In [ ]:
CONFIG = {
    "metric": "M5",
    "expected_rq": "RQ2",
    "unit_of_analysis": "global_transcript_corpus_by_temporal_marker",
    "source_contract": "data/lake/transcript_sessions.parquet",
    "primary_outputs": [
        "friction_marker_density_per_1k_tokens",
        "friction_subtype_composition",
        "transcript_corpus_coverage",
    ],
    "legacy_source": "paper_v4/advanced_metrics/outputs/cohort_temporal_friction.csv",
    "llm_calls_required": False,
    "inference": "descriptive_only",
    "cohort_coverage_requirement": "report_observed_semesters_without_imputation",
    "export_outputs": False,
}
REQUIRED_CONFIG_FIELDS = {
    "metric", "expected_rq", "unit_of_analysis", "source_contract",
    "primary_outputs", "legacy_source", "llm_calls_required", "inference",
    "cohort_coverage_requirement", "export_outputs",
}
assert not (REQUIRED_CONFIG_FIELDS - set(CONFIG))
assert CONFIG["expected_rq"] == detected_rq
assert CONFIG["llm_calls_required"] is False
CONFIG

## 2. Auditoria do M5 legado e da cobertura

O M5 publicado só reexpõe um CSV produzido por três chamadas LLM: uma por marco, após concatenar todo o texto disponível. O lake contém 100 **chunks técnicos de transcrição**, obtidos pela divisão das gravações para viabilizar o processamento inicial. Eles não são sessões adicionais, nem observações independentes.

A auditoria reproduz o artefato legado, mede sua variabilidade e declara a cobertura por chunks, sessão de origem e corpus antes de propor qualquer substituição.

In [ ]:
legacy_m5 = pd.read_csv(LEGACY_M5_PATH)
legacy_source_m5 = pd.read_csv(LEGACY_M5_SOURCE_PATH)
assert legacy_m5.equals(legacy_source_m5)
assert set(legacy_m5.columns) == {"Semestre", "temporal_marker", "coordination_friction"}
assert legacy_m5["Semestre"].eq("all").all()
assert legacy_m5["temporal_marker"].tolist() == ["T1", "T2", "T3"]

legacy_audit = pd.DataFrame([
    ("published_rows", len(legacy_m5)),
    ("distinct_scores", legacy_m5["coordination_friction"].nunique()),
    ("score_range", legacy_m5["coordination_friction"].max() - legacy_m5["coordination_friction"].min()),
    ("legacy_llm_calls", len(legacy_m5)),
    ("analysis_unit", "cohort_temporal_marker"),
], columns=["check", "value"])
display(legacy_m5)
legacy_audit

In [ ]:
transcripts = pd.read_parquet(TRANSCRIPTS_PATH).copy()
REQUIRED_TRANSCRIPT_COLUMNS = {
    "session_id", "Semestre", "temporal_marker", "transcript_text", "status",
}
missing_transcript_columns = REQUIRED_TRANSCRIPT_COLUMNS - set(transcripts.columns)
assert not missing_transcript_columns, sorted(missing_transcript_columns)
assert transcripts["status"].eq("success").all()
assert transcripts["transcript_text"].notna().all()
assert transcripts["transcript_text"].str.strip().ne("").all()
assert "ID_Equipe" not in transcripts.columns

coverage = (
    transcripts.groupby(["Semestre", "temporal_marker"], dropna=False)
    .agg(
        transcript_chunk_n=("transcript_file", "size"),
        source_session_n=("session_id", "nunique"),
        character_n=("transcript_text", lambda values: int(values.str.len().sum())),
        median_chunk_characters=("transcript_text", lambda values: float(values.str.len().median())),
    )
    .reset_index()
    .sort_values(["Semestre", "temporal_marker"])
)
observed_semesters = tuple(sorted(transcripts["Semestre"].unique()))
assert observed_semesters == ("2025.2",)
assert coverage["temporal_marker"].tolist() == ["T1", "T2", "T3"]
assert coverage["source_session_n"].eq(2).all()
assert int(coverage["transcript_chunk_n"].sum()) == len(transcripts)
coverage

## 3. Codificação reproduzível do corpus

A rubrica LLM legada mencionava bloqueios, mal-entendidos, repasses, conflitos de integração e sobrecarga recorrente, mas não preservava evidência por unidade. A substituição usa um léxico português versionado. Ele é uma medida de **evidência textual**, não diagnóstico de coordenação: resultados são descritivos e devem ser acompanhados pelos termos e contagens abaixo.

In [ ]:
FRICTION_LEXICON_VERSION = "m5-friction-lexicon-pt-v1"
FRICTION_LEXICON = {
    "alignment": r"coordena[çc][aã]o|comunica[çc][aã]o|alinh|sincron|divis[aã]o de taref|distribui[çc][aã]o de taref|depend[eê]ncia",
    "handoff": r"repasse|handoff|deleg[aã]|passar.{0,30}para",
    "integration": r"integra[çc][aã]o|merge|conflit|incompatibil|acopl",
    "blocker": r"bloque|imped|trava|gargalo",
    "rework": r"retrabalh|refaz|refazer|revis[aã]o",
}
assert set(FRICTION_LEXICON) == {"alignment", "handoff", "integration", "blocker", "rework"}
for pattern in FRICTION_LEXICON.values():
    re.compile(pattern)

coded_chunks = transcripts[["session_id", "Semestre", "temporal_marker", "transcript_text"]].copy()
coded_chunks["normalized_text"] = coded_chunks["transcript_text"].str.lower()
coded_chunks["token_n"] = coded_chunks["normalized_text"].map(lambda text: len(re.findall(r"\b\w+\b", text)))
for subtype, pattern in FRICTION_LEXICON.items():
    coded_chunks[subtype] = coded_chunks["normalized_text"].map(
        lambda text, expression=pattern: len(re.findall(expression, text))
    )
coded_chunks["friction_marker_n"] = coded_chunks[list(FRICTION_LEXICON)].sum(axis=1)
assert coded_chunks["token_n"].gt(0).all()
assert coded_chunks["friction_marker_n"].eq(coded_chunks[list(FRICTION_LEXICON)].sum(axis=1)).all()
# Chunks preserve transcription provenance only; they are never treated as independent observations.
coded_chunks[["session_id", "temporal_marker", "token_n", "friction_marker_n", *FRICTION_LEXICON]].head()

### 3.1 Trajetória global por marco

Cada marco é reconstruído em um único corpus pela concatenação de seus chunks técnicos. Densidade e composição são calculadas após essa reconstrução, sem tratar chunks ou sessões como replicações independentes. Como existem apenas dois áudios de origem por marco, não há intervalo de confiança, teste de hipótese nem média por sessão.

In [ ]:
corpus_by_marker = (
    coded_chunks.sort_values(["temporal_marker", "session_id"])
    .groupby("temporal_marker", sort=False)
    .agg(
        corpus_text=("normalized_text", "\n".join),
        transcript_chunk_n=("normalized_text", "size"),
        source_session_n=("session_id", "nunique"),
        token_n=("token_n", "sum"),
        friction_marker_n=("friction_marker_n", "sum"),
        **{subtype: (subtype, "sum") for subtype in FRICTION_LEXICON},
    )
    .reindex(["T1", "T2", "T3"])
    .reset_index()
)
corpus_by_marker["friction_marker_density_per_1k_tokens"] = (
    1_000 * corpus_by_marker["friction_marker_n"] / corpus_by_marker["token_n"]
)
assert corpus_by_marker["transcript_chunk_n"].sum() == len(coded_chunks)
assert corpus_by_marker["token_n"].sum() == coded_chunks["token_n"].sum()
assert corpus_by_marker["friction_marker_n"].sum() == coded_chunks["friction_marker_n"].sum()
assert corpus_by_marker["source_session_n"].eq(2).all()
corpus_by_marker[[
    "temporal_marker", "transcript_chunk_n", "source_session_n", "token_n",
    "friction_marker_n", "friction_marker_density_per_1k_tokens",
]]

### 3.2 Composição e evidência auditável

A composição identifica qual tipo de evidência cresce ou diminui no corpus. Para tornar o resultado revisável, o notebook também expõe os chunks de origem com maior número de marcadores; esses trechos servem para auditoria qualitativa, não para criar uma amostra estatística.

In [ ]:
subtype_columns = list(FRICTION_LEXICON)
subtype_composition = corpus_by_marker.melt(
    id_vars=["temporal_marker", "friction_marker_n"],
    value_vars=subtype_columns,
    var_name="friction_subtype",
    value_name="marker_n",
)
subtype_composition["marker_share"] = (
    subtype_composition["marker_n"] / subtype_composition["friction_marker_n"].replace(0, np.nan)
)
assert subtype_composition.groupby("temporal_marker")["marker_n"].sum().equals(
    corpus_by_marker.set_index("temporal_marker")["friction_marker_n"]
)

chunk_evidence_queue = (
    coded_chunks.loc[coded_chunks["friction_marker_n"].gt(0)]
    .sort_values(["friction_marker_n", "token_n"], ascending=False)
    [["temporal_marker", "session_id", "friction_marker_n", "token_n", *subtype_columns, "transcript_text"]]
    .reset_index(drop=True)
)
display(subtype_composition.sort_values(["temporal_marker", "marker_n"], ascending=[True, False]))
chunk_evidence_queue.head(10)

## 4. Adequação à RQ2 e limites de interpretação

A trajetória revisada preserva o eixo temporal necessário para RQ2: a densidade de evidência sobe de $0{,}515$ no T1 para $1{,}132$ no T2 e $1{,}287$ marcadores por mil tokens no T3. Ela contrasta qualitativamente com a atividade de repositório, mas não estima associação estatística com M3 ou M4.

Limites obrigatórios:

- todos os corpora observados pertencem a `2025.2`; não há base textual para comparar semestres;
- há duas gravações de origem por marco e 100 chunks técnicos no total; nem sessões nem chunks são réplicas independentes;
- o léxico identifica evidência textual candidata, não prova que a coordenação tenha falhado;
- não se deve aplicar teste de hipótese, intervalo de confiança ou correlação sobre os três marcos.

In [ ]:
m5_trajectory = corpus_by_marker[[
    "temporal_marker", "transcript_chunk_n", "source_session_n", "token_n",
    "friction_marker_n", "friction_marker_density_per_1k_tokens",
]].copy()
m5_trajectory["semester_coverage"] = ", ".join(observed_semesters)
m5_trajectory["lexicon_version"] = FRICTION_LEXICON_VERSION
m5_trajectory["analysis_level"] = CONFIG["unit_of_analysis"]
assert m5_trajectory["semester_coverage"].eq("2025.2").all()
assert m5_trajectory["analysis_level"].eq(CONFIG["unit_of_analysis"]).all()
m5_trajectory

## 5. Triagem de redundância

M5 não é redundante com as reformulações de M3 ou M4 porque observa uma modalidade e um construto distintos. Ainda assim, não se deve correlacioná-los: M3/M4 têm grão equipe-semestre/commit, enquanto M5 tem somente três corpora globais de `2025.2`. O contraste de RQ2 é descritivo e temporal, não uma análise de associação entre unidades incompatíveis.

In [ ]:
redundancy_triage = pd.DataFrame([
    {
        "metric": "M3a-M3c",
        "observed_modality": "Git commits and authors",
        "analytical_grain": "team-semester, commit, rolling window",
        "construct": "timing and authorship concentration",
        "overlap_with_m5": "none: no text or friction coding",
        "joint_analysis": "descriptive context only",
    },
    {
        "metric": "M4a-M4c",
        "observed_modality": "Git file changes",
        "analytical_grain": "team-semester and commit",
        "construct": "clean change magnitude, intensity, artifact composition",
        "overlap_with_m5": "none: no transcript content",
        "joint_analysis": "descriptive temporal contrast only",
    },
    {
        "metric": "M5a-M5c",
        "observed_modality": "Anonymized presentation transcripts",
        "analytical_grain": "global corpus by temporal marker",
        "construct": "textual evidence of coordination friction",
        "overlap_with_m5": "reference metric",
        "joint_analysis": "no correlation or causal claim",
    },
])
assert redundancy_triage["observed_modality"].nunique() == len(redundancy_triage)
assert redundancy_triage.loc[2, "joint_analysis"] == "no correlation or causal claim"
redundancy_triage

## 6. Decisão de refatoração e pipeline

**Decisão:** retirar o M5 legado como resultado analítico principal. Ele é reproduzível, mas três notas LLM constantes ($8,8,8$) não permitem descrever mudança, variabilidade ou evidência rastreável.

Adotar M5a--M5c somente como descrição do corpus global observado por marco, com o léxico, a versão e a cobertura registrados. Não executar novamente a pipeline congelada; quando o paper for regenerado, implementar esta transformação determinística em um script dedicado que consuma `transcript_sessions.parquet` e publique a tabela global por marco. Não fazer chamadas LLM.

In [ ]:
paper_correction_checklist = pd.DataFrame([
    ("definition", "Replace the single 1-10 LLM score with M5a-M5c and name the global corpus as the analysis level."),
    ("coverage", "State that all transcript evidence is from 2025.2, six source recordings, and 100 technical chunks."),
    ("claims", "Remove the claim that friction is constant at 8/10 across both cohorts."),
    ("comparison", "Describe M3/M4 versus M5 as a descriptive temporal contrast; do not report correlation, causality, or team-level association."),
    ("limitations", "State lexical false-positive/false-negative risk and the absence of team identifiers in transcript data."),
    ("provenance", "Publish lexicon version, marker counts, token denominators, subtype composition, and auditable source-chunk queue."),
], columns=["paper_area", "required_change"])
paper_correction_checklist

## 7. Checagens de regressão e manifesto de evidência

As verificações abaixo tornam explícitas as condições mínimas para executar o M5 revisado. Elas impedem mudanças silenciosas no vínculo com a RQ, no corpus, na reconstrução dos chunks, no léxico e no escopo inferencial.

In [ ]:
assert detected_rq == "RQ2"
assert legacy_m5["coordination_friction"].nunique() == 1
assert observed_semesters == ("2025.2",)
assert len(coded_chunks) == 100
assert corpus_by_marker["transcript_chunk_n"].sum() == len(coded_chunks)
assert corpus_by_marker["source_session_n"].eq(2).all()
assert corpus_by_marker["token_n"].sum() == coded_chunks["token_n"].sum()
assert corpus_by_marker["friction_marker_n"].sum() == coded_chunks["friction_marker_n"].sum()
assert corpus_by_marker[subtype_columns].sum(axis=1).equals(corpus_by_marker["friction_marker_n"])
assert not CONFIG["llm_calls_required"]
assert CONFIG["inference"] == "descriptive_only"

m5_evidence_manifest = {
    "metric": "M5",
    "rq": detected_rq,
    "analysis_level": CONFIG["unit_of_analysis"],
    "source_contract": str(TRANSCRIPTS_PATH.relative_to(PROJECT_ROOT)),
    "legacy_source": str(LEGACY_M5_SOURCE_PATH.relative_to(PROJECT_ROOT)),
    "observed_semesters": list(observed_semesters),
    "temporal_markers": corpus_by_marker["temporal_marker"].tolist(),
    "source_session_n_by_marker": dict(zip(corpus_by_marker["temporal_marker"], corpus_by_marker["source_session_n"])),
    "transcript_chunk_n_by_marker": dict(zip(corpus_by_marker["temporal_marker"], corpus_by_marker["transcript_chunk_n"])),
    "lexicon_version": FRICTION_LEXICON_VERSION,
    "llm_calls_required": CONFIG["llm_calls_required"],
    "inference": CONFIG["inference"],
}
m5_evidence_manifest